In [2]:
##modules
#%matplotlib widget
#%matplotlib inline
#
#%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

matplotlib.use('TkAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs
from statsmodels.tsa.stattools import acf

import numpy as np

import pandas as pd


from joblib import Parallel, delayed

import pickle

In [3]:

try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_SELF import get_paths_SELF

# Parámetros editables
disco = "g"
layer_script = "event"
subj = "s01b"


# Generar variables automáticamente
path_dict = get_paths_SELF(disco=disco, layer_script=layer_script, subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")


    

✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\ICA_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_clean_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_matlab_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_preproc\preproc_event\evoked_event
✅ Carpeta creada: g:\PROYECTO_SELF\channels_structure
✅ Carpeta creada: g:\PROYECTO_SELF\output_source\source_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_source\source_event\raw_hsp
✅ Carpeta creada: g:\PROYECTO_SELF\output_source\source_event\fwd
✅ Carpeta creada: g:\PROYECTO_SELF\output_source\source_event\inverse
✅ Carpeta creada: g:\PROYECTO_SELF\output_analysis\analysis_event\acw_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_analysis\analysis_event\PLE_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_analysis\analysis_event\ISC_event
✅ Carpeta creada: g:\PROYECTO_SELF\output_analysis\anal

In [4]:

pickle_file = epochs_clean_path / f"dict_conditions.pkl"
# Cargar el pickle
with open(pickle_file, "rb") as f:
    dict_conditions = pickle.load(f)

#epochs
combinaciones = list(dict_conditions.keys())
print(f"combinaciones: {combinaciones}")

# print(subj)
subjects = sorted({f.name.split("_")[0].lower() for f in data_task_edf.glob("*.edf")})
print(f"subj: {subjects}")


combinaciones: ['self_pos', 'self_neu', 'self_neg', 'friend_pos', 'friend_neu', 'friend_neg', 'unk_pos', 'unk_neu', 'unk_neg']
subj: ['s01b', 's02b', 's03b', 's04b', 's05b', 's06b', 's07b', 's08b', 's09b', 's10b', 's11b', 's12b', 's13b', 's14b', 's15b', 's16b', 's17b', 's18b', 's19b', 's20b', 's21b', 's22b', 's23b', 's24b', 's25b', 's26b', 's27b', 's28b', 's29b']


In [5]:

# channels = pd.read_csv(channels_structure_path / f"channels_eeg_{modality}.csv")
# channels_mag=channels[channels[f"canal_efectivo_{modality}"].notna()][f"canal_efectivo_{modality}"]
# channels_mag=channels_mag.tolist()
# del channels


edf_file = data_task_edf / f"{subjects[0]}_vis_c_BVica-export.edf"
elp_file = data_task_edf / f"{subjects[0]}_vis_c_BVica-export.elp"

# -------------------------
# 2. Leer EDF
# -------------------------

# Leer el EDF
raw = mne.io.read_raw_edf(edf_file, preload=True)
channels_eeg = mne.pick_types(raw.info, eeg=True, meg=False, eog=False, exclude=[])
channels_eeg_names = [raw.ch_names[i] for i in channels_eeg]

Extracting EDF parameters from F:\WORKAREA\Datos SELF\Self_Exp2\Exp2_review\Visual_Corregidos\ICA\EDF\s01b_vis_c_BVica-export.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 2465499  =      0.000 ...  4930.998 secs...


In [26]:
# aplicar la acf EN epochs


def acf_epochs(subj, epochs, condition, adjusted=False, fft=True, alpha=None, 
               bartlett_confint=True, missing="none", isplot=False, crop=None): 
    
    if crop is not None:
        epochs.crop(tmin=crop)  # recorta desde 'crop' segundos en adelante
    # Get only MEG channels, exclude bads
    data_epochs = epochs.copy().pick(picks="eeg", exclude="bads").get_data()
        # Si crop está definido, recortar desde 'crop' hasta el final

    
    channels_mag = epochs.copy().pick(picks="eeg", exclude="bads").ch_names

    # Get duration and sample freq
    duration = epochs.tmax - epochs.tmin
    sfreq = epochs.info['sfreq']
    lags = int(duration * sfreq)

    # Storage lists
    acf_elect_all_epoch_all = []
    acf_mean_elect_all_epoch_all = []
    acw_50_elect_all_epoch_all = []
    acw_0_elect_all_epoch_all = []

    # 1️⃣ Definir función paralela (fuera del bucle)
    def compute_acf_acw_sensor(data_sensor):
        acf_vals, qstat_vals, pvals = acf(
            data_sensor,
            adjusted=adjusted,
            fft=fft,
            qstat=True,
            nlags=lags,
            alpha=alpha,
            bartlett_confint=bartlett_confint,
            missing=missing
        )
        acw_50_lags = np.argmax(acf_vals <= 0.5)
        acw_50_s = acw_50_lags / sfreq
        acw_0_lags = np.argmax(acf_vals <= 0)
        acw_0_s = acw_0_lags / sfreq
        acf_mean = np.mean(acf_vals)
        return acf_vals, acf_mean, acw_50_s, acw_0_s

    # 2️⃣ Procesar cada época
    for j, data_epoch in enumerate(data_epochs):
        # Paralelizar sobre sensores en la época
        results = Parallel(n_jobs=-1)(
            delayed(compute_acf_acw_sensor)(data_epoch[i])
            for i in range(len(data_epoch))
        )

        # 3️⃣ Extraer resultados
        acf_elect_all_epoch = [r[0] for r in results]
        acf_mean_elect_all_epoch = [r[1] for r in results]
        acw_50_elect_all_epoch = [r[2] for r in results]
        acw_0_elect_all_epoch = [r[3] for r in results]

        acf_elect_all_epoch_all.append(acf_elect_all_epoch)
        acf_mean_elect_all_epoch_all.append(acf_mean_elect_all_epoch)
        acw_50_elect_all_epoch_all.append(acw_50_elect_all_epoch)
        acw_0_elect_all_epoch_all.append(acw_0_elect_all_epoch)

    # 🔄 Preparar resultados para DataFrame
    num_epochs = len(data_epochs)
    num_elects = len(channels_mag)
    shape_tabla = num_epochs * num_elects

    #acf_array = np.array(acf_elect_all_epoch_all)  # shape: (n_epochs, n_chans, n_lags)
    #acf_list = acf_array.reshape(shape_tabla, acf_array.shape[2]).tolist()

    df = pd.DataFrame({
        'Subject': [subj] * shape_tabla,
        'Condition': [condition] * shape_tabla,
        'Epoch': np.repeat(np.arange(num_epochs), num_elects),
        'Elect': np.tile(channels_mag, num_epochs),
        #'acf_elect_all_epoch_all': acf_list,
        'acw_50_elect_all_epoch_all': np.array(acw_50_elect_all_epoch_all).flatten(),
        'acw_0_elect_all_epoch_all': np.array(acw_0_elect_all_epoch_all).flatten()
    })

    return df


    isplot=False    

    # if isplot==True:
    #     # 1) Creamos un vector de lags en segundos,
    #     #    asumiendo que acf_val tiene tantos puntos como lags + 1.
    #     time_lags = np.arange(len(acf_val)) / sfreq

    #     # 2) Convertimos los índices (entero) de ACW_50 y ACW_0
    #     ACW_50_i = int(acw_50_lags)
    #     ACW_0_i = int(acw_0_lags)

    #     # 3) Figura
    #     plt.figure(figsize=(8, 4))

    #     # 4) Dibujamos la ACF en negro
    #     plt.plot(time_lags, acf_val, 'k', label='ACF')

    #     # 5) Acotamos el eje X a la duración total de la señal (o a lo que consideres)
    #     plt.xlim([0, time_lags[-1]])

    #     # 6) Rellenamos el área hasta ACW_50
    #     plt.fill_between(
    #         time_lags[:ACW_50_i + 1],
    #         acf_val[:ACW_50_i + 1],
    #         color='r', alpha=0.3, label='ACW-50 area'
    #     )

    #     # 7) Rellenamos el área hasta ACW_0
    #     plt.fill_between(
    #         time_lags[:ACW_0_i + 1],
    #         acf_val[:ACW_0_i + 1],
    #         color='m', alpha=0.3, label='ACW-0 area'
    #     )

    #     # 8) Título con valores de ACW
    #     plt.title(f'ACW-0 = {acw_0:.1f} s    ACW-50 = {acw_50:.1f} s')
    #     plt.xlabel('Lags (s)')
    #     plt.ylabel('Autocorrelation')
    #     plt.legend(loc='best')
    #     plt.show()

    return table_autocorrelation
    # return acf_epoch_all,acw_50_elect_all_epoch_all, acw_0_elect_all_epoch_all




In [ ]:
##codigo para agrupar todas las tablas

all_tables = []

# for i in range(0,len(subj)):
for i in range(0,len(subjects)):
    for h in range(0,len(combinaciones)):
        try:
            subj=subjects[i]
            combinacion= combinaciones[h]
            path_epochs= epochs_clean_path / f"{subj}_epochs_{combinacion}_{layer_script}-epo.fif"
            epochs = mne.read_epochs(path_epochs)
            table_autocorrelation= acf_epochs(subj, epochs, condition=combinacion,isplot=False,crop=0.5)
            all_tables.append(table_autocorrelation)
            del epochs
        except Exception as e:
            print(f"Error en {subj} {combinacion}: {e}")

autocorrelation_subjects_all = pd.concat(all_tables, ignore_index=True)


Reading g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_clean_event\s01b_epochs_self_pos_event-epo.fif ...
    Found the data of interest:
        t =    -500.00 ...    9000.00 ms
        0 CTF compensation matrices available
Not setting metadata
32 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_2208\3031852667.py:10: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs = epochs.copy().pick(picks="eeg", exclude="bads").get_data()


Reading g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_clean_event\s01b_epochs_self_neu_event-epo.fif ...
    Found the data of interest:
        t =    -500.00 ...    9000.00 ms
        0 CTF compensation matrices available
Not setting metadata
32 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_2208\3031852667.py:10: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs = epochs.copy().pick(picks="eeg", exclude="bads").get_data()


Reading g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_clean_event\s01b_epochs_self_neg_event-epo.fif ...
    Found the data of interest:
        t =    -500.00 ...    9000.00 ms
        0 CTF compensation matrices available
Not setting metadata
27 matching events found
No baseline correction applied
0 projection items activated


...

Reading g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_clean_event\s03b_epochs_unk_neu_event-epo.fif ...
    Found the data of interest:
        t =    -500.00 ...    9000.00 ms
        0 CTF compensation matrices available
Not setting metadata
26 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_2208\3031852667.py:10: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs = epochs.copy().pick(picks="eeg", exclude="bads").get_data()


Reading g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_clean_event\s03b_epochs_unk_neg_event-epo.fif ...
    Found the data of interest:
        t =    -500.00 ...    9000.00 ms
        0 CTF compensation matrices available
Not setting metadata
25 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_2208\3031852667.py:10: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs = epochs.copy().pick(picks="eeg", exclude="bads").get_data()


Error en s04b self_pos: File does not exist: "g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_clean_event\s04b_epochs_self_pos_event-epo.fif"
Error en s04b self_neu: File does not exist: "g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_clean_event\s04b_epochs_self_neu_event-epo.fif"
Error en s04b self_neg: File does not exist: "g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_clean_event\s04b_epochs_self_neg_event-epo.fif"
Error en s04b friend_pos: File does not exist: "g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_clean_event\s04b_epochs_friend_pos_event-epo.fif"
Error en s04b friend_neu: File does not exist: "g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_clean_event\s04b_epochs_friend_neu_event-epo.fif"
Error en s04b friend_neg: File does not exist: "g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_clean_event\s04b_epochs_friend_neg_event-epo.fif"
Error en s04b unk_pos: File does not exist: "g:\PROYECTO_SELF\output_preproc\preproc_event\epochs_clean_ev

In [10]:
autocorrelation_subjects_all

,Subject,Condition,Epoch,Elect,acw_50_elect_all_epoch_all,acw_0_elect_all_epoch_all
0,sub-V1001,zinnen,0,MLC11-4304,0.023333,0.346667
1,sub-V1001,zinnen,0,MLC12-4304,0.026667,0.350000
2,sub-V1001,zinnen,0,MLC13-4304,0.030000,0.366667
3,sub-V1001,zinnen,0,MLC14-4304,0.036667,0.363333
4,sub-V1001,zinnen,0,MLC15-4304,0.036667,0.350000
...,...,...,...,...,...,...
224674,sub-V1031,woorden,22,MZF03-4304,0.016667,0.230000
224675,sub-V1031,woorden,22,MZO01-4304,0.016667,0.036667
224676,sub-V1031,woorden,22,MZO02-4304,0.016667,0.033333
224677,sub-V1031,woorden,22,MZO03-4304,0.016667,0.030000


In [7]:
ACW_path

WindowsPath('g:/MOUS_204/MOUS_visual/output_analysis/analysis_block/acw_block')

In [29]:
autocorrelation_subjects_all.to_pickle(ACW_path / f"autocorrelation_subjects_all_{layer_script}.pickle")

In [30]:
acw_results_subjects_all= autocorrelation_subjects_all[['Subject', 'Condition', 'Epoch', 'Elect', 'acw_50_elect_all_epoch_all','acw_0_elect_all_epoch_all']]
acw_results_subjects_all.to_pickle(ACW_path / f"acw_results_subjects_all_{layer_script}.pickle")


In [25]:
acw_results_subjects_all= autocorrelation_subjects_all[['Subject', 'Condition', 'Epoch', 'Elect', 'acw_50_elect_all_epoch_all','acw_0_elect_all_epoch_all']]


In [31]:
acw_results_subjects_all

,Subject,Condition,Epoch,Elect,acw_50_elect_all_epoch_all,acw_0_elect_all_epoch_all
0,s01b,self_pos,0,Fp1,0.006,0.450
1,s01b,self_pos,0,Fpz,0.008,0.570
2,s01b,self_pos,0,Fp2,0.390,2.148
3,s01b,self_pos,0,AF7,0.330,1.848
4,s01b,self_pos,0,AF3,0.006,0.450
...,...,...,...,...,...,...
39584,s03b,unk_neg,24,PO4,0.210,0.566
39585,s03b,unk_neg,24,PO8,0.230,0.530
39586,s03b,unk_neg,24,O1,0.004,0.568
39587,s03b,unk_neg,24,Oz,0.004,0.470


Elects are : ['Fp1' 'Fpz' 'Fp2' 'AF7' 'AF3' 'AF4' 'AF8' 'F7' 'F5' 'F3' 'F1' 'Fz' 'F2'
 'F4' 'F6' 'F8' 'FT7' 'FC5' 'FC3' 'FC1' 'FCz' 'FC2' 'FC4' 'FC6' 'FT8' 'T7'
 'C5' 'C3' 'C1' 'Cz' 'C2' 'C4' 'C6' 'T8' 'TP7' 'CP5' 'CP3' 'CP1' 'CPz'
 'CP2' 'CP4' 'CP6' 'TP8' 'P7' 'P5' 'P3' 'P1' 'Pz' 'P2' 'P4' 'P6' 'P8'
 'PO7' 'PO3' 'PO4' 'PO8' 'O1' 'Oz' 'O2']
Epochs are : [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31]
